In [ ]:
# Install DeepChem and RDkit
!pip install -q deepchem
!pip install rdkit-pypi

In [ ]:

# Takes a dataset with smiles & labels cols and split into train set and test set
# using Bemis-Murcko Based scaffold splits 
import pandas as pd
import deepchem as dc
import os

#Step 1: Load the dataset
INPUT_CSV = "/kaggle/input/datasets/heisenbert/hdac1-v2-classified/HDAC_classifiedV2.csv"  # Make sure this file is in the Kaggle input directory
OUTPUT_DIR = "/kaggle/working"  # Default Kaggle output directory

df = pd.read_csv(INPUT_CSV)

# Drop rows with missing SMILES or labels
df = df.dropna(subset=["smiles", "labels"])

# Ensure SMILES are strings and labels are integers
df["smiles"] = df["smiles"].astype(str)
df["labels"] = df["labels"].astype(int)

print(f" Cleaned dataset with {len(df)} valid entries.")

# === Step 2: Convert to DeepChem Dataset ===
smiles_list = df["smiles"].tolist()
labels_list = df["labels"].tolist()

dc_dataset = dc.data.NumpyDataset(
    X=labels_list,          # dummy
    y=labels_list,
    ids=smiles_list         # SMILES strings
)

# === Step 3: Scaffold split ===
splitter = dc.splits.ScaffoldSplitter()
train_dataset, test_dataset = splitter.train_test_split(dc_dataset,frac_train=0.8,seed=42)

# === Step 4: Extract back to DataFrames ===
train_df = pd.DataFrame({
    "smiles": train_dataset.ids,
    "labels": train_dataset.y.flatten().astype(int)
})
test_df = pd.DataFrame({
    "smiles": test_dataset.ids,
    "labels": test_dataset.y.flatten().astype(int)
})

print(f" Scaffold Split: Train = {len(train_df)}, Test = {len(test_df)}")

# === Step 5: Save as CSVs ===
train_path = os.path.join(OUTPUT_DIR, "train_scffld.csv")
test_path = os.path.join(OUTPUT_DIR, "test_scffld.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f" Saved: {train_path}")
print(f" Saved: {test_path}")


In [ ]:
# Takes a dataset with smiles & labels cols and split into random train set and test set


import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split


# =====================================================
# Load Dataset
# =====================================================
# Replace 'your_dataset.csv' with your actual Kaggle dataset file path
df = pd.read_csv('/kaggle/input/datasets/heisenbert/caxii-classified/binary_dataset_caxii.csv')

print("Original dataset shape:", df.shape)
display(df.head())

# ------------------------------------------------------------
# Train-Test Split (Random 80/20)
# ------------------------------------------------------------
train_df, test_df = train_test_split(
    df,                    # <-- the DataFrame to split
    test_size=0.2,
    random_state=42,
    stratify=df['labels']  # optional but recommended for class balance
)

# If you also want X/y arrays (common for sklearn pipelines):
X_train, y_train = train_df['smiles'], train_df['labels']
X_test,  y_test  = test_df['smiles'],  test_df['labels']

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

# Save clean splits
train_df.to_csv("train_clean.csv", index=False)
test_df.to_csv("test_clean.csv", index=False)